In [2]:
import gzip
import json
import sys
from collections import Counter
import pandas as pd

In [9]:
path = "../data/abcd_v1.1.json.gz"

with gzip.open(path, "rt") as f:
    data = json.load(f) 

print(f"Conversations per Split: { {split: len(conversations) for split , conversations in data.items()}}")

Conversations per Split: {'train': 8034, 'dev': 1004, 'test': 1004}


In [15]:
# Sample
example = data["train"][0]

print(f"Intent (flow / subflow): {example['scenario']['flow']}")
print()
print(f"Subflow: {example['scenario']['subflow']}")
print("--- Conversations ---")
for speaker, message in example["original"]:
    print(f"{speaker : >9}: {message}")

Intent (flow / subflow): product_defect

Subflow: return_size
--- Conversations ---
    agent: Hi!
    agent: How can I help you?
 customer: Hi! I need to return an item, can you help me with that?
    agent: sure, may I have your name please?
 customer: Crystal Minh
    agent: thanks, may I ask the reason for the return?
   action: Account has been pulled up for Crystal Minh.
 customer: I got the wrong size.
    agent: ok, may I have your username, email address and order ID please?
 customer: Username: cminh730
 customer: cminh730@email.com
 customer: Order ID: 3348917502
   action: Purchase validation in progress ...
    agent: thanks so much! What is your membership level Crystal?
 customer: I'm a bronze
    agent: ok, was the purchase made in the last 90 days?
 customer: No, I bought it in November.
    agent: ok, unfortunately because it has been more than 90 days we cannot accept the return. Would there be anything else I can help you with?
 customer: What if I ask really, reall

In [14]:
flow_count = Counter(c["scenario"]["flow"] for c in data["train"])
print("Conversations per intent (Training data)")

for flow, n in flow_count.most_common():
    print(f"{flow}: {n}")

Conversations per intent (Training data)
storewide_query: 872
product_defect: 863
purchase_dispute: 858
account_access: 847
single_item_query: 840
order_issue: 831
troubleshoot_site: 819
shipping_issue: 814
subscription_inquiry: 718
manage_account: 572


In [24]:
def agent_button(conversation):
    """Return the action button list that the agent clicked on in a conversation."""
    return [
        turn["targets"][2] for turn in conversation["delexed"]
        if turn["speaker"] == "action" and len(turn["targets"]) > 2
    ]

def bot_solvable(conversation):
    """Return True if the conversation is solvable by the bot, False otherwise."""
    buttons = agent_button(conversation)
    return bool(buttons) and all(b.startswith("search-") or b == "select-faq" for b in buttons)



In [25]:
rows = []
for split, conversations in data.items():
    for c in conversations:
        
        first_msg = next((t for s, t in c["original"] if s == "customer"), "")

        rows.append({
            "conversation_id": c["convo_id"],
            "split": split,
            "flow": c["scenario"]["flow"],
            "subflow": c["scenario"]["subflow"],
            "first_customer_message": first_msg,
            "bot_solvable": bot_solvable(c),
        })

df = pd.DataFrame(rows)
print("Split of Bot-solvable conversations per Intent (Training Set):")
print(
    df[df.split == "train"]
    .groupby("flow")["bot_solvable"].mean()
    .sort_values(ascending=False)
    .map("{:.0%}".format)
)

print(f"\nOverrall bot-solvable conversations: {df.bot_solvable.mean():.0%}")

Split of Bot-solvable conversations per Intent (Training Set):
flow
single_item_query       90%
storewide_query         82%
manage_account           2%
purchase_dispute         1%
shipping_issue           1%
subscription_inquiry     0%
troubleshoot_site        0%
account_access           0%
product_defect           0%
order_issue              0%
Name: bot_solvable, dtype: str

Overrall bot-solvable conversations: 19%


In [26]:
df.to_csv("../data/abcd_v1.1_summary.csv", index=False)
print(f"\n saved abcd scenarios to csv! length: {len(df)} rows")


 saved abcd scenarios to csv! length: 10042 rows
